# Differential expression analysis
2025-09-30  
Data are EP1NS cells (ependymoma ZFTA-fusion cell line) treated with shL1CAM, shSHTN1, shYAP1, or shControl.
[DESeq2 docs](https://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html).

## Required input files:
*shEP1NS.expected_counts.tsv*.  
Generated by running RSEM on all samples using the following parameters:  
`rsem-calculate-expression --star --star-gzipped-read-file --paired-end --append-names $R1 $R2 $REF $SAMPLE`  
where the reference is ensembl 113 (GRCH38.p14)

*sample_metadata.tsv*.  
Sample metadata indicating shRNA treatments. Formatted as tsv of sample, treatment.

*Homo_sapiens.GRCh38.113.gtf*.  
Gene annotation file used for STAR, RSEM. Download from https://ftp.ensembl.org/pub/release-113/gtf/homo_sapiens/Homo_sapiens.GRCh38.113.gtf.gz.

## Results


In [ ]:
# Load dependencies

Sys.setenv(LANGUAGE = "en") # set language to "ja" if you prefer
suppressWarnings(library(DESeq2))
suppressWarnings(library(plyranges))
suppressWarnings(library(dplyr))
suppressWarnings(library(tibble))
suppressWarnings(library(readr))
suppressWarnings(library(ggplot2))
suppressWarnings(library(ggrepel))
suppressWarnings(library(extrafont))
suppressWarnings(library(svglite))
suppressWarnings(library(patchwork)) # combine plots

suppressMessages(extrafont::font_import(pattern="Arial",prompt=FALSE))
suppressMessages(extrafont::loadfonts())

getwd()
sessionInfo()

In [ ]:
# Plotting defaults
base_theme <- theme_classic(base_size=14, base_family="Arial",) +
    theme(axis.text = element_text(size=14,colour="black"),
          aspect.ratio=1,
          #plot.margin=unit(c(0,0,0,0), "null")
         )
theme_set(base_theme)

write_plot <- function(plt,outfile,width,height){
    b=basename(outfile)
    d=dirname(outfile)
    dir.create(d, recursive=TRUE, showWarnings = FALSE)
    pdf.options(encoding='ISOLatin2.enc')
    #pdfName = paste(outfile, ".pdf", sep="")
    pngName = paste(b, ".png", sep="")
    svgName = paste(b, ".svg", sep = "")
    #ggsave(path="figures", filename=pdfName, device="pdf", width=width, height=height, units='in')
    ggsave(path=d, device="png", filename=pngName, width=width, height=height, units='in')
    ggsave(path=d, device="svg", filename=svgName, width=width, height=height, units='in')

}

# Get and process your data

In [ ]:
data_location = 'results/rsem/shEP1NS.expected_counts.tsv'
metadata_location = 'anno/sample_metadata.tsv'
annotation_location = 'anno/ensembl-113/Homo_sapiens.GRCh38.113.gtf'

In [ ]:
load_annotations <- function(annotation_path) {
    # Load  gene annotations from a .gtf file.
    # I'm using the basic primary gene annotation file from ensembl 113:
    # https://ftp.ensembl.org/pub/release-113/gtf/homo_sapiens/Homo_sapiens.GRCh38.113.gtf.gz
    g <- rtracklayer::import(annotation_path) %>%
        filter(type=='gene') %>%
       filter(gene_biotype=='protein_coding')
    return(g)
}

filter_protein_coding <- function(gene_matrix,annotation_path){
    # given a matrix with rownames of the form ENSG00000000003_TSPAN6,
    # return a subset consisting of only rows where the stable gene ID ENSG00000000003 
    # is annotated as a protein-coding gene.
    message('filtering for protein-coding genes ...')
    g <- load_annotations(annotation_path)
    # need to match on stable ENSG IDs
    ensembl_ids <- sub("_.*", "", rownames(gene_matrix))
    keep <- ensembl_ids %in% g$gene_id
    filtered_data <- gene_matrix[keep, ]
    return(filtered_data)
}

load_gex  <- function(data_location,annotation_location){
    message("loading gene expression data from ", data_location, "...")
    cts = as.matrix(read.csv(data_location,sep='\t',row.names="gene_id",check.names=FALSE))
    cts = round(cts) # counts must be integer
    cts = filter_protein_coding(cts,annotation_location)
    new_rownames <- sub("^ENSG\\d*_", "", rownames(cts)) # Remove ENSG prefixes
    rownames(cts) <- new_rownames
    return(cts)
}

In [ ]:
# Read and preprocess expression data
data = load_gex(data_location,annotation_location)

In [ ]:
# Read and format our annotation table
annot = read.table(metadata_location, row.names=1, header=TRUE)
annot <- annot[match(colnames(data), rownames(annot)), , drop=FALSE]
annot <- annot %>%
    mutate(batch = factor(batch))

# Drop possibly problematic controls
# drop = c("shNC_REP3","shNC_REP4")
# data = data[, setdiff(colnames(data),drop)]
# annot = annot[setdiff(rownames(annot),drop), , drop=FALSE]

data %>% head
annot

# Full linear model on all samples


In [ ]:
dds <- DESeq2::DESeqDataSetFromMatrix(countData = data,
                              colData = annot,
                              design = ~ batch + treatment)
# Pre-filter
smallestGroupSize <- 2
keep <- rowSums(counts(dds) >= 30) >= smallestGroupSize
dds <- dds[keep,]
# Run regression
dds <- DESeq(dds)
# Print model summary
dds
# print model coefficient names
resultsNames(dds)

In [ ]:
# Looks like much more RNA usable from all 150bp reads than 75bp.
sizeFactors(dds)

In [ ]:
# PCA on all samples. First PC is read length. Probably discard batches 3 and 4 for YAP1 analysis.
vsd <- vst(dds, blind=TRUE)

a = plotPCA(vsd, intgroup=c("treatment")) + geom_label_repel(aes(label = name))
w=8;h=8
write_plot(a,outfile='results/deseq2/shYAP1/pca',width=w,height=h)
options(repr.plot.width=w, repr.plot.height=h)
a

# Linear model on batches 1&2 only

In [ ]:
## Subset on batch
anno2 = annot %>% 
    filter(batch %in% c(1,2))
data2 = data[,rownames(anno2)]
data2 %>% head
anno2

In [ ]:
dds2 <- DESeq2::DESeqDataSetFromMatrix(countData = data2,
                              colData = anno2,
                              design = ~ batch + treatment)
# Pre-filter
smallestGroupSize <- 2
keep <- rowSums(counts(dds2) >= 30) >= smallestGroupSize
dds2 <- dds2[keep,]
# Run regression
dds2 <- DESeq(dds2)
# Print model summary
dds2
# print model coefficient names
resultsNames(dds2)

In [ ]:
# Write library size normalized counts to file
file = "results/deseq2/shYAP1/shYAP1.deseq2norm.tsv"
counts(dds2,normalized=TRUE) %>% write.table(
  file = file,
  sep = "\t",
  quote = FALSE,
  col.names = NA
)

In [ ]:
## PCA on batch 1&2 samples. 
## PC1 would separate treatments but YAP1_REP1-3 is an outlier (65% variance explained). 
## PC2 separates batches (20% variance explained).

vsd <- vst(dds2, blind=TRUE)

a = plotPCA(vsd, intgroup=c("treatment")) + geom_label_repel(aes(label = name))
w=8;h=8
write_plot(a,outfile='results/deseq2/shYAP1/pca_batch1-2',width=w,height=h)
options(repr.plot.width=w, repr.plot.height=h)
a

In [ ]:
## Generate DE comparisons. Wald test with BH correction. Post-hoc LFC shrinkage with apeglm.
res2 <- results(dds2, name="treatment_shYAP1_vs_control",independentFiltering=TRUE)
res2 <- lfcShrink(dds2, coef="treatment_shYAP1_vs_control", res=res2, type="apeglm")


In [ ]:
# genes down in shYAP1 
#get_hits(res2,FALSE,outfile='results/deseq2/shYAP1/shYAP1_batch1-2_deg.tsv') %>% head(n=10)

In [ ]:
# Lookup genes of interest
genes_of_interest = c('SHTN1','L1CAM','CCN2','CCN1','YAP1','BIRC5','ITGAV') # CTGF = CCN2; CYR61 =  CCN1

res2[rownames(res2) %in% genes_of_interest,]

# Linear model on batches 1&2 only

In [ ]:
## Subset batches
# Drop outliers
drop = c("shYAP1_REP1-3")
anno3 = annot %>% 
    filter(batch %in% c(1,2)) %>%
    filter(!rownames(.) %in% drop)
data3 = data[,rownames(anno3)]
data3 %>% head
anno3

In [ ]:
dds3 <- DESeq2::DESeqDataSetFromMatrix(countData = data3,
                              colData = anno3,
                              design = ~ batch + treatment)
# Pre-filter
smallestGroupSize <- 2
keep <- rowSums(counts(dds3) >= 30) >= smallestGroupSize
dds3 <- dds3[keep,]
# Run regression
dds3 <- DESeq(dds3)
# Print model summary
dds3
# print model coefficient names
resultsNames(dds3)

In [ ]:
# Write library size normalized counts to file
file = "results/deseq2/shYAP1/shYAP1.deseq2norm.tsv"
counts(dds3,normalized=TRUE) %>% write.table(
  file = file,
  sep = "\t",
  quote = FALSE,
  col.names = NA
)

In [ ]:
## PCA on batch 1&2 samples less outlier sample YAP1_REP1-3. 
## PC1  separates treatments (62% variance explained). 
## PC2 separates batches (23% variance explained).

vsd <- vst(dds3, blind=TRUE)

a = plotPCA(vsd, intgroup=c("treatment")) + geom_label_repel(aes(label = name))
w=8;h=8
write_plot(a,outfile='results/deseq2/shYAP1/pca_batch1-2_drop-outlier',width=w,height=h)
options(repr.plot.width=w, repr.plot.height=h)
a

In [ ]:
## Generate DE comparisons. Wald test with BH correction. Post-hoc LFC shrinkage with apeglm.
res3 <- results(dds3, name="treatment_shYAP1_vs_control",independentFiltering=TRUE)
res3 <- lfcShrink(dds3, coef="treatment_shYAP1_vs_control", res=res3, type="apeglm")


In [ ]:
# Save DEGs

get_hits <- function(deseq_result,decreasing=FALSE,outfile=NULL){
    res_df <- as.data.frame(deseq_result)  # Convert DESeqResults object to a data frame
    res_df <- res_df[order(res_df$log2FoldChange, decreasing=decreasing),]  # Sort by foldChange
    # Filter by padj < 0.05
    filtered_sorted_res <- res_df[(!is.na(res_df$padj)) & (res_df$padj < 0.05), ]  # Filter by padj
    
    if (!is.null(outfile)) {
        # Write the data frame to a tab-separated file (TSV)
        write.table(res_df, file = outfile, sep = "\t", quote = FALSE)
    }
    return(filtered_sorted_res)
}

get_hits(res3,FALSE,outfile='results/deseq2/shYAP1/shYAP1_batch1-2_deg.tsv') %>% head(n=10)

In [ ]:
# Lookup genes of interest
genes_of_interest = c('SHTN1','L1CAM','CCN2','CCN1','YAP1','BIRC5','ITGAV') # CTGF = CCN2; CYR61 =  CCN1

res3[rownames(res3) %in% genes_of_interest,]

# Make a volcano plot

In [ ]:
Sys.setenv(LANGUAGE = "en") # set language to "ja" if you prefer
suppressWarnings(library(EnhancedVolcano))

In [ ]:
res3 = read.delim('results/deseq2/shYAP1/shYAP1_batch1-2_deg.tsv')

In [ ]:
## Plotting code

write_plot <- function(plt,outfile,width,height,path="results/deseq2/shYAP1"){
    pdf.options(encoding='ISOLatin2.enc')
    #pdfName = paste(outfile, ".pdf", sep="")
    pngName = paste(outfile, ".png", sep="")
    svgName = paste(outfile, ".svg", sep = "")
    #ggsave(path="figures", filename=pdfName, device="pdf", width=width, height=height, units='in')
    ggsave(path=path, device="png", filename=pngName, width=width, height=height, units='in')
    ggsave(path=path, device="svg", filename=svgName, width=width, height=height, units='in')

}

ylabel=expression(-Log[10]*"("*italic(q)*")")

base_theme <- theme_classic(base_size=7, base_family="Arial",) +
    theme(axis.text = element_text(size=7,colour="black"))
theme_set(base_theme)


In [ ]:
genes_of_interest = c('SHTN1','L1CAM','CCN2','CCN1','YAP1','BIRC5') # CTGF = CCN2; CYR61 =  CCN1

osc_volcano_v <- function(stats_df,genes_of_interest){
    stats_df$highlight = ifelse(rownames(stats_df) %in% genes_of_interest,"notable","other")
    stats_df <- stats_df[order(stats_df$highlight=='other',decreasing=TRUE),]
    stats_df$color <- mapply(function(fc, padj, gene) {
      if (gene %in% genes_of_interest) {
        "blue4"  # custom color for notable
      } else if (fc < -1 & padj < 0.05) {
        "dodgerblue"   # upper-left
      } else if (fc > 1 & padj < 0.05) {
        "red"    # upper-right
      } else {
        "grey50" # everything else
      }
    }, stats_df$log2FoldChange, stats_df$padj, rownames(stats_df))

    plt <- EnhancedVolcano(stats_df,
                lab = rownames(stats_df),
                title = NULL,
                subtitle = NULL,
                caption = NULL,
                axisLabSize = 14,
                #x = 'logFC',
                #y = "adj.P.Val",
                #xlim = c(-2.75,2.75),
                x = 'log2FoldChange',
                y = 'padj',
                #ylim = c(0,4),
                pCutoff = 0.05,
                selectLab = genes_of_interest,
                #labSize = 4.21644413212,#3.37315530569,#
                #FCcutoff = 10,
                #vline = NULL, 
                #vlineType = "blank",
                legendPosition = "none",
                pointSize = c(ifelse(stats_df$highlight == "other", 1, 3)),
                colCustom = setNames(stats_df$color,rownames(stats_df)),
                drawConnectors = TRUE,
                maxoverlapsConnectors = Inf,
                lengthConnectors = unit(0, "npc"),
                colAlpha = c(ifelse(stats_df$highlight == "other", .6, .8)),
                ) %>% suppressWarnings
    options(repr.plot.width=7, repr.plot.height=7)
    return(plt + ylab(ylabel))
}

In [ ]:
options(warn=-1)
plt <- osc_volcano_v(res3,c('CCN2','CCN1','YAP1','BIRC5'))
w=5;h=5
options(repr.plot.width=2*w, repr.plot.height=2*h)
write_plot(plt,"selected_volcano",w,h)
plt
options(warn=0)


In [ ]:
res3$log2FoldChange